## Imports

In [1]:
import os
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account


## Setup

Authentication is handled via a Google Cloud service account.
The credentials file is stored locally and excluded from the repository via `.gitignore`.

In [2]:
# Path service account credentials file
CREDENTIALS_FILE = "final-project-dla-00934ab13a3c.json"
PROJECT_ID = "final-project-dla"

# Authenticate
credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_FILE)

# Initialize BigQuery client
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

print(f"Connected to project: {PROJECT_ID}")


Connected to project: final-project-dla


In [3]:
def run_query(file_name: str) -> pd.DataFrame:
    """
    Reads a SQL file from the sql/ folder and executes it via BigQuery client.

    Args:
        file_name (str): Path relative to the sql/ folder,
                         e.g. '1_data_quality/01_table_row_counts.sql'

    Returns:
        pd.DataFrame: Query result as a pandas DataFrame.

    Raises:
        FileNotFoundError: If the specified SQL file does not exist.
    """
    base_path = os.path.join(os.getcwd(), 'sql')
    full_path = os.path.join(base_path, file_name)

    if not os.path.exists(full_path):
        raise FileNotFoundError(f"Query file not found: {full_path}")

    with open(full_path, 'r') as f:
        query = f.read()

    print(f"Executing: {file_name}")
    return client.query(query).to_dataframe()

# Retail Customer Basket Analysis
## Data Quality & Validation

Before answering any analytical questions, we need to validate the dataset: check table sizes, null values, duplicates, and join coverage between tables. This block ensures our analysis is built on reliable data.

### 1. Row counts across all tables

In [4]:
run_query('1_data_quality/01_table_row_counts.sql')

Executing: 1_data_quality/01_table_row_counts.sql


,table_name,row_count
0,hh_demographic,801
1,product,92353
2,transaction_data,2595732
3,campaign_table,7208
4,coupon_redempt,2318


**Findings:**
- `transaction_data` is the largest table with **2,595,732 rows** - this is our main fact table
- `product` table contains **92,353** unique products
- `hh_demographic` covers **801** households
- `coupon_redempt` has **2,318** redemption records across **7,208** campaigns
- All 5 tables are present and populated - no empty tables detected

### 2. Transaction data overview

In [5]:
run_query('1_data_quality/02_transaction_data_overview.sql')

Executing: 1_data_quality/02_transaction_data_overview.sql


,total_rows,unique_customers,unique_baskets,first_day,last_day,observation_window_days,total_revenue,avg_transaction_value
0,2595732,2500,276484,1,711,710,8057463.08,3.1


**Findings:**
- Dataset spans **711 days** of shopping activity (approximately 2 years)
- **2,500** unique households and **276,484** unique baskets recorded
- Total revenue across the observation period: **$8,057,463**
- Average transaction line value is **$3.10** - this is per item row, not per basket
- The dataset is large enough and time-rich enough to support reliable behavioral analysis

### 3. Null check in transaction data

In [6]:
run_query('1_data_quality/03_null_check_transactions.sql')

Executing: 1_data_quality/03_null_check_transactions.sql


,total_rows,null_household_key,null_basket_id,null_day,null_sales_value,null_quantity,null_product_id
0,2595732,0,0,0,0,0,0


**Findings:**
- No null values detected in any critical field
- The transaction table is fully populated and ready for analysis

### 4. Null check in demographic data

In [7]:
run_query('1_data_quality/04_demographic_null_check.sql')

Executing: 1_data_quality/04_demographic_null_check.sql


,households_with_demographics,null_age,null_income,null_household_size,null_marital_status
0,801,0,0,0,0


**Findings:**
- Demographic table contains **801 households** with no missing values
- **Note**: only 801 out of 2,500 unique households have demographic data - this coverage limitation will be important to keep in mind when interpreting segmentation results, as they represent a subset of the full customer base

### 5. Coupon table overview

In [8]:
run_query('1_data_quality/05_coupon_table_overview.sql')

Executing: 1_data_quality/05_coupon_table_overview.sql


,total_redemptions,unique_households,unique_campaigns
0,2318,434,30


**Findings:**
- **2,318** total coupon redemptions recorded
- **434** unique households redeemed at least one coupon
- **30** unique campaigns in the dataset
- **Note**: only 434 out of 2,500 unique transacting households used coupons - approximately **17%** of the customer base.

### 6. Duplicate check in transaction data

In [9]:
run_query('1_data_quality/06_duplicate_check_transactions.sql')

Executing: 1_data_quality/06_duplicate_check_transactions.sql


,total_duplicate_rows
0,0


**Findings:**
- The transaction table is clean and requires no deduplication before analysis

### 7. Duplicate check in demographic data

In [10]:
run_query('1_data_quality/07_duplicate_check_demographics.sql')

Executing: 1_data_quality/07_duplicate_check_demographics.sql


,household_key,occurrences


**Findings:**
- The demographic table is clean and requires no deduplication

### 8. Zero and negative value analysis

In [11]:
run_query('1_data_quality/08_zero_negative_value_analysis.sql')

Executing: 1_data_quality/08_zero_negative_value_analysis.sql


,negative_sales,zero_sales,zero_or_negative_quantity,min_sales_value,max_sales_value,max_quantity,zero_sales_and_zero_qty,zero_sales_positive_qty,positive_sales_non_positive_qty,total_rows
0,0,18850,14466,0.0,840.0,89638,14399,4451,67,2595732


**Findings:**
- **No negative sales values** detected - a good sign, no refund/return anomalies
- **18,850 zero-value sales rows** found - representing transactions where products were recorded but no revenue was captured (likely free samples, promotional items, or data entry issues)
- **14,466 zero or negative quantity rows** detected
- Breaking down the overlap:
    - **14,399 rows** have both zero sales and zero/negative quantity - these are likely fully invalid records
    - **4,451 rows** have zero sales but positive quantity - products recorded with no revenue attached
    - **67 rows** have positive sales but non-positive quantity - a small but notable inconsistency
- `max_sales_value` of **$840** and `max_quantity` of **89,638** suggest extreme outliers that will need investigation in the business questions block  

*All of these anomalies are handled by filtering them out in the `transactions_clean` and `basket_summary` views, ensuring the analysis is based on valid records only*

### 9. Demographic join coverage

In [12]:
run_query('1_data_quality/09_demographic_join_coverage.sql')

Executing: 1_data_quality/09_demographic_join_coverage.sql


,total_households,households_with_demographics,demographic_coverage_pct
0,2500,801,32.04


**Findings:**
- **801 out of 2,500** transacting households have demographic data
- Demographic coverage is **32.04%** - meaning segmentation analysis covers roughly one third of the full customer base
- However, 801 households still provides a sufficient sample for meaningful behavioral comparisons across demographic segments

### 10. Product join coverage

In [13]:
run_query('1_data_quality/10_product_join_coverage.sql')

Executing: 1_data_quality/10_product_join_coverage.sql


,total_products_in_transactions,matched_products_in_product_table,unmatched_products
0,92339,92339,0


**Findings:**
- **92,339 out of 92,339** products in transactions matched to the product reference table - **100% join coverage**

### 11. Coupon household coverage

In [14]:
run_query('1_data_quality/11_coupon_household_coverage_pct.sql')

Executing: 1_data_quality/11_coupon_household_coverage_pct.sql


,total_households,coupon_households,coupon_household_pct
0,2500,434,17.36


**Findings:**
- **434 out of 2,500** transacting households redeemed at least one coupon
- Coupon coverage is **17.36%** of the customer base
- This gives us two clearly defined groups for campaign analysis: **434 coupon households** vs **2,066 non-coupon households**
- The groups are imbalanced in size but large enough for a meaningful behavioral comparison

# Block 1 - Retail Performance

This block answers the three core retail performance questions:
- Which product categories generate the highest revenue and transaction volume?
- What does the basket value distribution look like?
- What is the average number of items per basket, and which categories 
dominate large baskets?

All queries in this block use the `transactions_clean` and `basket_summary` views, which excludes zero-value rows and other anomalies identified in the data quality block.

### Q1. Which product categories generate the highest revenue and transaction volume?

In [15]:
run_query('3_analytical_questions/01_category_revenue.sql')

Executing: 3_analytical_questions/01_category_revenue.sql


,category,total_baskets,total_revenue,revenue_pct,avg_item_value
0,GROCERY,215173,4093810.84,50.81,2.49
1,DRUG GM,117793,1055357.16,13.10,3.83
2,PRODUCE,88923,557437.89,6.92,2.18
3,MEAT,53334,548786.49,6.81,6.23
4,KIOSK-GAS,22056,544222.28,6.75,24.67
5,MEAT-PCKGD,58385,412436.77,5.12,3.72
6,DELI,35560,260865.94,3.24,4.17
7,PASTRY,30047,121739.86,1.51,3.25
8,MISC SALES TRAN,5932,119960.04,1.49,20.04
9,NUTRITION,16781,97669.04,1.21,3.04


**Findings:**
- **GROCERY** dominates with **$4,093,810** - **50.81%** of total revenue, appearing in more baskets than any other category by a wide margin
- **DRUG GM**, **PRODUCE**, **MEAT**, and **KIOSK-GAS** form a clear middle tier, each contributing **~6–13%** of revenue
- **KIOSK-GAS** and **MISC SALES TRAN** show unusually high avg item values (**$24.67** and **$20.04**) - reflecting non-standard transaction types that will be excluded from item count analysis
- The top 5 categories account for **~84%** of total revenue — a highly concentrated revenue structure typical of grocery retail

**Key insight:** Half of all revenue comes from GROCERY alone. Promotional and category strategy should prioritize GROCERY, DRUG GM, 
PRODUCE, and MEAT for maximum business impact.

### Q2. What does the basket value distribution look like, and what are the average and median basket values?

In [16]:
run_query('3_analytical_questions/02_basket_value_stats.sql')

Executing: 3_analytical_questions/02_basket_value_stats.sql


,avg_basket_value,stddev_basket_value,min_basket_value,max_basket_value,p25,median_basket_value,p75,p90
0,29.24,36.12,0.01,961.49,6.99,17.18,36.39,70.78


In [17]:
run_query('3_analytical_questions/03_basket_value_buckets.sql')

Executing: 3_analytical_questions/03_basket_value_buckets.sql


,basket_size_bucket,basket_count,pct_of_baskets,avg_value_in_bucket
0,1. Under $5,50683,18.39,2.63
1,2. $5–$10,41822,15.18,7.37
2,3. $10–$20,58230,21.13,14.44
3,4. $20–$30,38756,14.07,24.40
4,5. $30–$50,39976,14.51,38.40
5,6. $50–$75,21074,7.65,60.91
6,7. $75–$100,10636,3.86,86.27
7,8. $100+,14362,5.21,145.76


**Findings:**
- **Average basket value is $29.24** but the **median is only $17.18** - a significant gap indicating right skew pulled up by large baskets
- **Standard deviation of $36.12** is higher than the mean itself, confirming high variability in shopping behavior
- **90% of baskets are under $70.78** - the $100+ segment represents only **5.21%** of baskets but likely contributes disproportionately to revenue
- Distribution is heavily concentrated in the **$10–$20 range (21.13%)**, with **~54% of all baskets falling under $20** - typical of frequent 
top-up shopping trips rather than large weekly shops
- The **max basket value of $961.49** confirms the presence of extreme outliers at the top end

**Key insight:** The typical shopper makes small, frequent trips - the median basket of $17.18 tells a more accurate story than the mean. 
High-value baskets ($100+) are rare but worth understanding as a separate customer behavior segment.

### Q3. What is the average number of items per basket, and which categories appear most often in large baskets?

#### Step 1: Initial exploration — items per basket (all categories)


In [18]:
run_query('3_analytical_questions/04_items_per_basket_raw.sql')

Executing: 3_analytical_questions/04_items_per_basket_raw.sql


,avg_items_per_basket,stddev_items,min_items,max_items,p25,median_items,p75,p90
0,946.08,3425.84,1.0,89638.0,3.0,8.0,22.0,69.0


**Initial findings:**
- **Average of 946 items per basket** is clearly unrealistic - heavily distorted by extreme outliers
- However, the **median of 8 items** and **p90 of 69** look reasonable and suggest the underlying data is sound
- Max of **89,638** confirms extreme values exist - investigation required

#### Step 2: Investigating extreme quantities

In [19]:
run_query('3_analytical_questions/05_extreme_quantity_investigation.sql')

Executing: 3_analytical_questions/05_extreme_quantity_investigation.sql


,product_id,department,commodity_desc,max_quantity,avg_quantity,avg_price_per_unit,row_count
0,6534178,KIOSK-GAS,COUPON/MISC ITEMS,89638,10930.43,0.0023,19810
1,6544236,MISC SALES TRAN,COUPON/MISC ITEMS,85055,19890.64,0.0026,136
2,6534166,MISC SALES TRAN,COUPON/MISC ITEMS,36140,11659.76,0.0024,1199
3,6533889,MISC SALES TRAN,COUPON/MISC ITEMS,32035,12727.00,0.0025,1452
4,5716076,KIOSK-GAS,FUEL,30080,23855.00,0.0026,2
5,5668996,KIOSK-GAS,COUPON/MISC ITEMS,29945,7592.26,0.0024,42
6,1404121,KIOSK-GAS,COUPON/MISC ITEMS,25750,9628.43,0.0024,176
7,1426702,MISC SALES TRAN,COUPON/MISC ITEMS,25652,12951.23,0.0025,35
8,397896,KIOSK-GAS,COUPON/MISC ITEMS,25567,10952.93,0.0023,122
9,6410462,KIOSK-GAS,COUPON/MISC ITEMS,22451,10554.25,0.0024,20


In [20]:
run_query('3_analytical_questions/06_extreme_rows_detail.sql')

Executing: 3_analytical_questions/06_extreme_rows_detail.sql


,household_key,basket_id,product_id,department,commodity_desc,quantity,sales_value,price_per_unit,DAY
0,630,34749153595,6534178,KIOSK-GAS,COUPON/MISC ITEMS,89638,250.00,0.0028,503
1,2407,29392047893,6544236,MISC SALES TRAN,COUPON/MISC ITEMS,85055,210.00,0.0025,181
2,630,29484790880,6534178,KIOSK-GAS,COUPON/MISC ITEMS,61335,150.21,0.0024,185
3,149,28210551971,6534178,KIOSK-GAS,COUPON/MISC ITEMS,51912,110.00,0.0021,103
4,193,32956767959,6534178,KIOSK-GAS,COUPON/MISC ITEMS,48073,121.10,0.0025,402
5,2133,41904760458,6534178,KIOSK-GAS,COUPON/MISC ITEMS,45475,100.00,0.0022,682
6,149,33768630428,6534178,KIOSK-GAS,COUPON/MISC ITEMS,41833,115.00,0.0027,456
7,149,29035716247,6534178,KIOSK-GAS,COUPON/MISC ITEMS,41686,100.00,0.0024,158
8,1406,28167562655,6544236,MISC SALES TRAN,COUPON/MISC ITEMS,41485,85.00,0.0020,99
9,107,27865225627,6534178,KIOSK-GAS,COUPON/MISC ITEMS,39365,72.00,0.0018,74


**Investigation findings:**
- All extreme quantities come exclusively from **KIOSK-GAS** and **MISC SALES TRAN** departments
- These are **fuel and coupon/misc transaction** records where quantity likely represents fuel units (e.g. milliliters or points) rather than 
physical item counts - not comparable to regular retail items
- Average price per unit of **~$0.002** confirms these are not standard retail products
- These two categories must be excluded for any meaningful item count analysis
- One **PRODUCE/CORN** entry with quantity 144 appears at the bottom - likely a legitimate bulk purchase and not an anomaly


#### Step 3: Adjusted items per basket — excluding KIOSK-GAS and MISC SALES TRAN

In [21]:
run_query('3_analytical_questions/07_items_per_basket_adjusted.sql')

Executing: 3_analytical_questions/07_items_per_basket_adjusted.sql


,avg_items_per_basket,stddev_items,min_items,max_items,p25,median_items,p75,p90
0,13.32,17.16,1.0,350.0,3.0,7.0,17.0,34.0


**Findings:**
- After excluding KIOSK-GAS and MISC SALES TRAN, the average drops dramatically from **946 to just 13.32 items per basket** - a far more 
realistic figure
- **Median of 7 items** confirms most shopping trips are small and focused
- **P90 of 34 items** was used as the threshold for defining "large baskets" in the next query
- Remaining max of **350 items** may still reflect some bulk purchases but is within a plausible retail range

#### Step 4: Which categories appear most often in large baskets?

In [22]:
run_query('3_analytical_questions/08_large_basket_category_composition.sql')

Executing: 3_analytical_questions/08_large_basket_category_composition.sql


,category,appearances_in_large_baskets,pct_of_large_baskets,avg_item_value
0,GROCERY,59998,22.59,2.47
1,PRODUCE,45343,17.07,2.23
2,DRUG GM,39893,15.02,3.66
3,MEAT-PCKGD,34289,12.91,3.84
4,MEAT,30979,11.66,6.44
5,DELI,17534,6.60,4.23
6,PASTRY,13172,4.96,2.92
7,NUTRITION,9473,3.57,3.12
8,SEAFOOD-PCKGD,5954,2.24,5.65
9,COSMETICS,2494,0.94,4.10


**Findings:**
- **GROCERY dominates large baskets** appearing in **22.59%** of them - consistent with its overall revenue leadership
- **PRODUCE** is notably stronger in large baskets (**17.07%**) than its revenue share (**6.92%**) suggests - large basket shoppers buy significantly more fresh produce
- **MEAT** and **MEAT-PCKGD** together account for **~24.5%** of large basket appearances - protein is a key driver of big shopping trips
- The large basket category ranking closely mirrors the overall revenue ranking, confirming that big spenders shop across the same core categories

**Key insight for Q3:** The typical basket contains **7 items**, but large basket shoppers consistently load up on GROCERY, PRODUCE, and MEAT - 
suggesting these categories should be prioritized for bulk promotions and loyalty rewards targeting high-value customers.

# Block 2 - Customer Segmentation

This block examines how shopping behavior differs across demographic segments:
- Age groups
- Income groups
- Household composition (presence of children)
- Customer loyalty tiers

All segmentation queries are based on the **801 households** with demographic data - representing **32%** of the full customer base.

### Q4. How does spending behavior differ across age groups?

In [23]:
age_order = ['19-24', '25-34', '35-44', '45-54', '55-64', '65+']

df_age = run_query('3_analytical_questions/09_spending_by_age_group.sql')
df_age['age_group'] = pd.Categorical(df_age['age_group'], categories=age_order, ordered=True)
df_age.sort_values('age_group')

Executing: 3_analytical_questions/09_spending_by_age_group.sql


,age_group,total_households,total_baskets,total_revenue,avg_basket_value,median_basket_value,avg_revenue_per_household,avg_visits_per_household
5,19-24,46,8113,216583.02,26.70,15.00,4708.33,176.37
0,25-34,142,22621,777934.53,34.39,21.14,5478.41,159.30
1,35-44,194,36496,1242144.89,34.04,20.07,6402.81,188.12
2,45-54,288,51241,1657466.34,32.35,19.76,5755.09,177.92
3,55-64,59,10012,298229.02,29.79,19.30,5054.73,169.69
4,65+,72,11388,305353.81,26.81,18.20,4241.03,158.17


**Findings:**
- **35-44** is the highest-value age group — highest avg revenue per household (**$6,402**), and most visits per household (**188**)
- **25-34** has the highest avg basket value (**$34.39**) despite being a smaller group - they spend more per trip but visit less frequently
- Spending power clearly **peaks in the 35-44 range** and gradually declines with age - the 65+ group has the lowest avg basket value 
(**$26.81**) and fewest visits (**158**)
- The **45-54** group has the most households (**288**) and generates the highest total revenue (**$1,657,466**) - driven by group size 
rather than individual spend
- **19-24** has the lowest median basket value (**$15.00**) - consistent with lower disposable income at this life stage
- The gap between mean and median is consistent across all groups, confirming right skew exists within every age segment

**Key insight:** The 35-44 age group is the most valuable customer segment - they shop most frequently and spend the most per household. 
Marketing and loyalty efforts should prioritize this group while developing strategies to increase visit frequency among 25-34 shoppers 
who already demonstrate high per-trip spending.

### Q5. Which income groups generate the highest revenue and largest average baskets?

In [24]:
income_order = [
    'Under 15K', '15-24K', '25-34K', '35-49K', '50-74K',
    '75-99K', '100-124K', '125-149K', '150-174K', '175-199K', '200-249K', '250K+'
]

df_income = run_query('3_analytical_questions/10_spending_by_income_group.sql')
df_income['income_group'] = pd.Categorical(df_income['income_group'], categories=income_order, ordered=True)
df_income.sort_values('income_group')

Executing: 3_analytical_questions/10_spending_by_income_group.sql


,income_group,total_households,total_baskets,total_revenue,avg_basket_value,median_basket_value,avg_revenue_per_household,avg_visits_per_household
9,Under 15K,61,12723,339135.49,26.66,15.81,5559.60,208.57
10,15-24K,74,12097,303187.20,25.06,15.32,4097.12,163.47
11,25-34K,77,15190,380230.49,25.03,13.24,4938.06,197.27
8,35-49K,172,29524,825944.84,27.98,16.84,4802.00,171.65
6,50-74K,192,32134,1094850.89,34.07,21.83,5702.35,167.36
5,75-99K,96,14262,558961.81,39.19,25.62,5822.52,148.56
7,100-124K,34,6329,201543.15,31.84,22.60,5927.74,186.15
4,125-149K,38,7625,300658.10,39.43,23.51,7912.06,200.66
0,150-174K,30,4930,251845.22,51.08,34.06,8394.84,164.33
3,175-199K,11,2092,94033.09,44.95,30.75,8548.46,190.18


**Findings:**
- Clear positive relationship between income and basket value - avg basket value rises steadily from **$25.03** (25-34K) to **$51.08** (150-174K)
- **250K+** households show the highest revenue per household (**$10,789**) and most visits (**217/year**) - highest engagement overall
- **150-174K**, **200-249K**, and **250K+** all cluster above **$49 avg basket value** - forming a clear high-income, high-spend tier
- **50-74K** is the largest group (**192 households**) and generates the most total revenue (**$1,094,850**) - the backbone of store revenue
- Interestingly, **Under 15K** households visit frequently (**208 visits/year**) despite low basket values (**$26.66**) - compensating with trip frequency
- **100-124K** breaks the income-spend pattern slightly with a lower avg basket value (**$31.84**) than the 75-99K group (**$39.19**) - 
worth noting as an anomaly

**Key insight:** Income is a strong predictor of basket value but not always of visit frequency. The mid-income 50-74K segment drives the most 
total revenue through volume, while high-income households ($150K+) contribute disproportionately per visit.

### Q6. Do households with children shop differently from those without - in terms of spend, basket size, and category preferences?

In [25]:
run_query('3_analytical_questions/11_spending_by_kid_composition.sql')

Executing: 3_analytical_questions/11_spending_by_kid_composition.sql


,kid_category,total_households,total_baskets,total_revenue,avg_basket_value,median_basket_value,avg_revenue_per_household,avg_visits_per_household
0,3+,69,11318,465168.96,41.10,26.64,6741.58,164.03
1,2,60,11161,405840.04,36.36,20.27,6764.00,186.02
2,1,114,20273,669720.57,33.04,18.96,5874.74,177.83
3,None/Unknown,558,97119,2956982.04,30.45,19.26,5299.25,174.05


In [26]:
run_query('3_analytical_questions/12_category_preferences_by_kid_composition.sql')

Executing: 3_analytical_questions/12_category_preferences_by_kid_composition.sql


,kid_category,category,total_revenue,pct_of_group_revenue,revenue_rank
0,1,GROCERY,343390.86,51.27,1
1,1,DRUG GM,93650.19,13.98,2
2,1,MEAT,44796.20,6.69,3
3,1,KIOSK-GAS,44642.87,6.67,4
4,1,PRODUCE,42243.61,6.31,5
5,1,MEAT-PCKGD,33236.46,4.96,6
6,1,DELI,19598.79,2.93,7
7,1,NUTRITION,9309.04,1.39,8
8,1,MISC SALES TRAN,8739.33,1.30,9
9,1,PASTRY,8193.97,1.22,10


**Findings:**

**Spending behavior:**
- Clear pattern - more children means higher basket value:
  **3+ kids ($41.10)** - **2 kids ($36.36)** - **1 kid ($33.04)** - **None/Unknown ($30.45)**
- Revenue per household is highest for **2-kid ($6,764)** and **3+ kid ($6,742)** households - larger families are the most valuable individual customers
- **None/Unknown** dominates total revenue (**$2,956,982**) purely due to group size (**558 households**) - not individual spend

**Important note on None/Unknown:** This category does not necessarily mean households without children - it may also include households where child composition data was not collected or not disclosed. Conclusions about child-free households should be treated with caution.

**Category preferences:**
- **GROCERY** is the top category across all groups at **~50%** of revenue - no meaningful difference here
- Households with **1 child** spend notably more on **MEAT (6.69%)** vs **2-child (5.69%)** and **3+ child (5.49%)** households - larger families shift toward packaged meat (MEAT-PCKGD) for cost efficiency
- **PRODUCE** share increases with more children — **3+ kids (7.71%)** vs **1 kid (6.31%)** - larger families buy more fresh food
- **KIOSK-GAS** is notably higher for **None/Unknown (7.59%)** and **2-kid (9.73%)** households - possibly reflecting more car-dependent 
shopping patterns

**Key insight:** Households with children are significantly more valuable per customer - spending more per basket and generating higher revenue per household. Promotions targeting family-sized products in GROCERY, PRODUCE, and MEAT would resonate most with this segment.

### Q7. Which households shop most frequently, and do the most loyal customers also have the highest basket values?

In [27]:
run_query('3_analytical_questions/13_top_households_by_spend.sql')

Executing: 3_analytical_questions/13_top_households_by_spend.sql


,household_key,total_visits,total_spend,avg_basket_value,max_basket_value,min_basket_value,first_visit_day,last_visit_day,active_days
0,1023,601,38319.79,63.76,543.83,1.39,107,710,603
1,1609,408,27859.68,68.28,961.49,1.69,42,711,669
2,2322,323,23646.92,73.21,298.51,1.89,66,711,645
3,1453,761,21661.29,28.46,334.10,0.34,97,710,613
4,2459,970,20671.50,21.31,384.72,0.25,35,704,669
5,1430,343,20352.99,59.34,280.40,1.33,76,711,635
6,718,599,19299.86,32.22,335.17,0.34,1,707,706
7,707,497,19194.42,38.62,254.46,0.59,103,711,608
8,1653,538,19153.75,35.60,205.33,0.68,90,710,620
9,1111,321,18894.72,58.86,269.55,0.59,32,707,675


In [28]:
run_query('3_analytical_questions/14_loyalty_tier_segmentation.sql')

Executing: 3_analytical_questions/14_loyalty_tier_segmentation.sql


,loyalty_tier,total_households,avg_visits,avg_total_spend,avg_basket_value,tier_total_revenue,pct_of_total_revenue
0,3. High Loyalty,851,221.8,6090.41,29.51,5182941.00,64.32
1,2. Mid Loyalty,831,78.0,2605.48,33.32,2165149.93,26.87
2,1. Low Loyalty,818,26.8,867.18,32.42,709352.87,8.80


In [29]:
run_query('3_analytical_questions/15_loyalty_basket_correlation.sql')

Executing: 3_analytical_questions/15_loyalty_basket_correlation.sql


,corr_visits_vs_avg_basket,corr_visits_vs_total_spend
0,-0.1244,0.703


**Findings:**

**Top households:**
- The most visited household (**#2459**) made **970 trips** but has a relatively low avg basket of **$21.31** - a frequent but small-basket shopper
- The highest spender (**#1023**) combines high frequency (**601 visits**) with a solid avg basket (**$63.76**) - the most balanced high-value customer
- Household **#1864** stands out with only **163 visits** but the highest avg basket value of **$103.94** - a rare but high-value shopper

**Loyalty tiers:**
- **High Loyalty** households (851) generate **64.32%** of total revenue despite being roughly equal in size to other tiers - driven by visit 
frequency (**222 visits/year**) rather than basket size
- Interestingly, **Mid and Low Loyalty** tiers actually have slightly **higher avg basket values** ($33.32 and $32.42) than High Loyalty ($29.51) - frequent shoppers make smaller, more routine trips

**Correlation:**
- **Weak negative correlation (-0.12)** between visit frequency and avg basket value - confirms that more frequent shoppers tend to spend less per trip
- **Strong positive correlation (0.70)** between visit frequency and total spend - frequency is the primary driver of overall customer value

**Key insight:** Loyalty is driven by visit frequency, not basket size. The most valuable customers shop often and consistently rather than 
spending large amounts per trip. Retention strategies should focus on encouraging repeat visits over upselling basket size.

# Block 3 - Campaign Impact

This block examines whether coupon redemption is associated with meaningfully different spending behavior:
- Do coupon households spend more per basket?
- Do they visit more frequently?
- How much of total revenue do they represent?

### Q8. Do households that redeemed coupons show meaningfully different spending patterns compared to those who did not?

In [30]:
run_query('3_analytical_questions/16_coupon_vs_noncoupon_spending.sql')

Executing: 3_analytical_questions/16_coupon_vs_noncoupon_spending.sql


,customer_type,total_households,avg_visits,avg_total_spend,avg_basket_value,median_basket_value,total_revenue,pct_of_total_revenue
0,Non-Coupon,2066,93.0,2510.36,29.99,25.99,5186397.02,64.37
1,Coupon,434,192.4,6615.32,39.98,35.20,2871046.78,35.63


**Findings:**
- Coupon households have a significantly higher avg basket value (**$39.98**) vs non-coupon households (**$29.99**) - a **33% difference**
- Coupon households visit more than twice as frequently (**192 vs 93 visits/year**) and spend more than double per household (**$6,615 vs $2,510**)
- Despite being only **17.36%** of the customer base, coupon households generate **35.63%** of total revenue - punching well above their weight
- The median basket value gap (**$35.20 vs $25.99**) confirms this difference is not driven by outliers - coupon shoppers genuinely spend more per trip

**Important caveat:** This is a correlation, not causation. Coupon households may already be more engaged, higher-spending customers who also happen to use coupons - rather than coupons causing higher spend. The direction of this relationship will be explored further in Block 5.

**Key insight:** Coupon users are the store's most engaged and valuable customers. The campaign program appears to attract and retain high-frequency, high-value shoppers - making it a strong candidate for further investment.

# Block 4 - Category Preferences Across Segments

This block examines whether different demographic segments favor clearly different product categories:
- Do age groups shop differently at the category level?
- Do income groups show distinct category preferences?

### Q9. Which product categories are most popular across different demographic segments - do income or age groups favor clearly different categories?

In [31]:
run_query('3_analytical_questions/17_top_categories_by_age_group.sql')

Executing: 3_analytical_questions/17_top_categories_by_age_group.sql


,age_group,category,total_revenue,pct_of_group_revenue,category_rank
0,19-24,GROCERY,117369.45,54.19,1
1,19-24,DRUG GM,29591.75,13.66,2
2,19-24,MEAT,13954.89,6.44,3
3,19-24,MEAT-PCKGD,12474.03,5.76,4
4,19-24,KIOSK-GAS,11783.70,5.44,5
5,25-34,GROCERY,397364.17,51.08,1
6,25-34,DRUG GM,104590.21,13.44,2
7,25-34,KIOSK-GAS,63622.35,8.18,3
8,25-34,PRODUCE,53315.29,6.85,4
9,25-34,MEAT,43754.81,5.62,5


In [32]:
income_order = [
    'Under 15K', '15-24K', '25-34K', '35-49K', '50-74K',
    '75-99K', '100-124K', '125-149K', '150-174K', '175-199K', '200-249K', '250K+'
]

df_categories_income = run_query('3_analytical_questions/18_top_categories_by_income_group.sql')
df_categories_income['income_group'] = pd.Categorical(
    df_categories_income['income_group'], 
    categories=income_order, 
    ordered=True
)
df_categories_income.sort_values('income_group')

Executing: 3_analytical_questions/18_top_categories_by_income_group.sql


,income_group,category,total_revenue,pct_of_group_revenue,category_rank
59,Under 15K,PRODUCE,23941.04,7.06,5
55,Under 15K,GROCERY,171450.61,50.56,1
58,Under 15K,KIOSK-GAS,24225.86,7.14,4
57,Under 15K,MEAT,30210.65,8.91,3
56,Under 15K,DRUG GM,38600.12,11.38,2
13,15-24K,KIOSK-GAS,19649.13,6.48,4
12,15-24K,MEAT,24069.93,7.94,3
11,15-24K,DRUG GM,41325.94,13.63,2
10,15-24K,GROCERY,153795.65,50.73,1
14,15-24K,MEAT-PCKGD,18088.87,5.97,5


**Findings:**

**By age group:**
- **GROCERY and DRUG GM** are ranked #1 and #2 across every single age group without exception - category preferences are remarkably consistent across ages
- The main difference is in positions 3-5: younger groups (19-24, 25-34, 35-44) 
have **KIOSK-GAS** ranked higher, while older groups (45-54, 55-64, 65+) favor **PRODUCE** over gas - possibly reflecting different mobility and 
lifestyle patterns
- **19-24** is the only group where **PRODUCE** doesn't appear in the top 5 at all - youngest shoppers prioritize packaged meat over fresh produce

**By income group:**
- Again **GROCERY and DRUG GM** dominate across all income groups at #1 and #2 - Higher income groups ($125K+) consistently show stronger **PRODUCE** share and weaker **KIOSK-GAS** presence - higher earners spend more on fresh food and less on fuel as a proportion of grocery spend
- Lower income groups (Under 15K, 15-24K, 25-34K) show stronger **MEAT** share in top 3 positions - possibly prioritizing protein staples
- **175-199K** and **250K+** are the only groups where **DELI** appears in the top 5, replacing KIOSK-GAS - higher income households spend more 
on prepared foods

**Key insight:** Category preferences are structurally very similar across all demographic segments - GROCERY and DRUG GM always lead. The meaningful differences appear in the 3rd-5th positions, reflecting lifestyle and income-driven shifts between PRODUCE, MEAT, KIOSK-GAS, and DELI.

# Summary

The dataset is clean and reliable with one key limitation - only **32% of households have demographic data**, meaning segmentation findings 
apply to a subset of the full customer base.

Core findings across all blocks:
- The store is fundamentally a grocery retailer - **GROCERY alone accounts for ~51% of revenue** across all segments
- The typical customer makes **small, frequent trips** - median basket of $17.18 and 7 items
- **Income and family size** are the strongest predictors of basket value
- **Visit frequency** is the primary driver of overall customer value - not basket size
- **Coupon households** are the most engaged customers - higher spend, higher frequency, disproportionate revenue contribution
- Category preferences are **remarkably consistent** across demographics, with meaningful differences only in positions 3-5

## Business Recommendations

**1. Double down on the core revenue engine**  
GROCERY and DRUG GM together account for approximately 64% of total revenue and appear in every demographic segment's top 2. Promotional investment, shelf space, and loyalty rewards should be concentrated here first before expanding to peripheral categories.

**2. Target families with children for high-value basket growth**  
Households with 2+ children spend 35% more per basket than the average. Bundle promotions in GROCERY, MEAT, and PRODUCE - the categories that over-index for this segment - would likely drive basket size growth among the store's most valuable individual customers.

**3. Prioritize visit frequency over basket size for loyalty programs**  
High loyalty customers drive 64% of total revenue through frequency, not spend per trip. Loyalty program design should reward visit consistency - stamps, visit-based tiers, or weekly incentives - rather than focusing solely on spend thresholds.

**4. Invest in the coupon program**  
Coupon households generate 35% of revenue from just 17% of customers. This makes coupon engagement a strong area for further investment, especially among mid-loyalty households. However, since this analysis is observational, future testing would be useful to determine whether broader coupon exposure directly increases spend or mainly attracts already high-value shoppers.

## Next Steps  
The next notebook **`02_visualizations_and_statistical_validation.ipynb`** extends this exploratory SQL analysis with visual validation and statistical validation of basket value differences across income groups.